### UMAP and t-SNE

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap.umap_ as umap

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

embedding_3d_dir = os.path.join(base_dir, "embedding_data", "3dembedding_data")
plots_dir = os.path.join(base_dir, "plots", "tsne_umap_from_3d")
embeddings_dir = os.path.join(base_dir, "embedding_data", "tsne_umap_from_3d")

os.makedirs(plots_dir, exist_ok=True)
os.makedirs(embeddings_dir, exist_ok=True)

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

# UMAP parameters
n_components_umap = 2
n_neighbors_umap = 15
min_dist_umap = 0.1
random_state_umap = 42

# t-SNE parameters
n_components_tsne = 2
perplexity_tsne = 30
random_state_tsne = 42

# Speed controls
# UMAP uses full data
# t-SNE uses a reproducible 10k-point subsample
tsne_max_points = 10000

# Plotting controls
# UMAP plot may still be subsampled for faster plotting only
plot_max_points_umap = 15000

# Plot style
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
})

rng = np.random.default_rng(42)

# -------------------------------------------------------
# STYLE HELPER
# -------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.20, color=ACCENT)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

# -------------------------------------------------------
# DATA HELPERS
# -------------------------------------------------------
def standardize_columns(X):
    X = np.asarray(X, dtype=float)
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True) + 1e-12
    return (X - mu) / sd

def subsample_rows(X, max_points=15000, rng=None):
    X = np.asarray(X)
    if rng is None:
        rng = np.random.default_rng()

    n = len(X)
    if max_points is None or n <= max_points:
        return X

    idx = np.sort(rng.choice(n, size=max_points, replace=False))
    return X[idx]

# -------------------------------------------------------
# MANIFOLD METHODS
# -------------------------------------------------------
def apply_umap(data, n_components=2, n_neighbors=15, min_dist=0.1, random_state=42):
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state,
        transform_seed=random_state,
        low_memory=True
    )
    return reducer.fit_transform(data)

def apply_tsne(data, n_components=2, perplexity=30, random_state=42):
    tsne = TSNE(
        n_components=n_components,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        method="barnes_hut",
        angle=0.5
    )
    return tsne.fit_transform(data)

# -------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------
summary_rows = []

for channel_name in eeg_channel_names:
    data_path = os.path.join(embedding_3d_dir, f"3dembedded_{channel_name}.npy")

    if not os.path.exists(data_path):
        print(f"Missing file for {channel_name}: {data_path}")
        continue

    channel_data_3d = np.load(data_path)
    channel_data_3d = np.asarray(channel_data_3d, dtype=float)

    if channel_data_3d.ndim != 2 or channel_data_3d.shape[1] != 3:
        print(f"Skipping {channel_name}: expected shape (N, 3), got {channel_data_3d.shape}")
        continue

    # Standardize before manifold learning
    X = standardize_columns(channel_data_3d)

    print(f"\nProcessing {channel_name} | full shape={X.shape}")

    # -------------------------
    # UMAP on full data
    # -------------------------
    t0 = time.perf_counter()
    umap_embedding = apply_umap(
        X,
        n_components=n_components_umap,
        n_neighbors=n_neighbors_umap,
        min_dist=min_dist_umap,
        random_state=random_state_umap
    )
    umap_time = time.perf_counter() - t0

    # -------------------------
    # t-SNE on 10k-point subsample
    # -------------------------
    X_tsne = subsample_rows(X, max_points=tsne_max_points, rng=rng)

    t0 = time.perf_counter()
    tsne_embedding = apply_tsne(
        X_tsne,
        n_components=n_components_tsne,
        perplexity=perplexity_tsne,
        random_state=random_state_tsne
    )
    tsne_time = time.perf_counter() - t0

    # -------------------------
    # SAVE EMBEDDINGS
    # -------------------------
    umap_path = os.path.join(embeddings_dir, f"umap_embedding_{channel_name}.npy")
    tsne_path = os.path.join(embeddings_dir, f"tsne_embedding_{channel_name}.npy")

    np.save(umap_path, umap_embedding)
    np.save(tsne_path, tsne_embedding)

    # -------------------------
    # PLOT UMAP (subsample for plotting only)
    # -------------------------
    umap_plot = subsample_rows(umap_embedding, max_points=plot_max_points_umap, rng=rng)

    fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
    style_ax(ax)
    ax.scatter(
        umap_plot[:, 0], umap_plot[:, 1],
        s=5, color=ACCENT
    )
    ax.set_title(f"UMAP for {channel_name} | n={X.shape[0]}")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"UMAP_{channel_name}.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # -------------------------
    # PLOT t-SNE (already subsampled to 10k)
    # -------------------------
    fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
    style_ax(ax)
    ax.scatter(
        tsne_embedding[:, 0], tsne_embedding[:, 1],
        s=5, color=ACCENT
    )
    ax.set_title(f"t-SNE for {channel_name} | n={X_tsne.shape[0]}")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"tSNE_{channel_name}.png"), dpi=150, bbox_inches="tight")
    plt.close()

    summary_rows.append({
        "channel": channel_name,
        "n_points_full": X.shape[0],
        "n_points_used_for_umap": X.shape[0],
        "n_points_used_for_tsne": X_tsne.shape[0],
        "umap_time_sec": umap_time,
        "tsne_time_sec": tsne_time,
        "umap_file": os.path.basename(umap_path),
        "tsne_file": os.path.basename(tsne_path),
    })

    print(
        f"Saved {channel_name} | "
        f"UMAP: {umap_embedding.shape} in {umap_time:.2f}s | "
        f"t-SNE: {tsne_embedding.shape} in {tsne_time:.2f}s"
    )

# -------------------------------------------------------
# SAVE SUMMARY
# -------------------------------------------------------
if len(summary_rows) > 0:
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(embeddings_dir, "tsne_umap_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"\nSaved summary CSV: {summary_csv}")

print("\nAll processes completed successfully.")

### 4D to 10D TMAP and t-SNE

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap.umap_ as umap

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

embedding_in_dir = os.path.join(base_dir, "embedding_data", "embeddings_4to10")
embeddings_out_dir = os.path.join(base_dir, "embedding_data", "tsne_umap_4to10")
plots_out_dir = os.path.join(base_dir, "plots", "tsne_umap_4to10")

os.makedirs(embeddings_out_dir, exist_ok=True)
os.makedirs(plots_out_dir, exist_ok=True)

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

emb_dims = list(range(4, 11))  # 4..10 inclusive

# UMAP parameters
n_components_umap = 2
n_neighbors_umap = 15
min_dist_umap = 0.1
random_state_umap = 42

# t-SNE parameters
n_components_tsne = 2
perplexity_tsne = 30
random_state_tsne = 42

# For speed:
# - full embeddings are still computed on all rows by default
# - only plotting is subsampled
plot_max_points = 15000

# If runtime becomes too large, set compute_max_points to an integer like 15000 or 20000.
# Keeping it None preserves full fidelity for the manifold computation.
compute_max_points = None

# Plot style
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
})

rng = np.random.default_rng(42)

# -------------------------------------------------------
# STYLE HELPER
# -------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.20, color=ACCENT)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

# -------------------------------------------------------
# HELPERS
# -------------------------------------------------------
def standardize_columns(X):
    X = np.asarray(X, dtype=float)
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True) + 1e-12
    return (X - mu) / sd

def subsample_rows(X, max_points=15000, rng=None):
    X = np.asarray(X)
    if rng is None:
        rng = np.random.default_rng()

    n = len(X)
    if max_points is None or n <= max_points:
        return X

    idx = np.sort(rng.choice(n, size=max_points, replace=False))
    return X[idx]

def apply_umap(data, n_components=2, n_neighbors=15, min_dist=0.1, random_state=42):
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state,
        transform_seed=random_state,
        low_memory=True
    )
    return reducer.fit_transform(data)

def apply_tsne(data, n_components=2, perplexity=30, random_state=42):
    tsne = TSNE(
        n_components=n_components,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        method="barnes_hut",
        angle=0.5
    )
    return tsne.fit_transform(data)

# -------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------
summary_rows = []

for emb_dim in emb_dims:
    print(f"\n==================== m = {emb_dim} ====================")

    for channel_name in eeg_channel_names:
        in_path = os.path.join(embedding_in_dir, f"{emb_dim}dembedded_{channel_name}.npy")

        if not os.path.exists(in_path):
            print(f"Missing: {in_path}")
            continue

        X = np.load(in_path)
        X = np.asarray(X, dtype=float)

        if X.ndim != 2 or X.shape[1] != emb_dim:
            print(f"Skipping {channel_name}, m={emb_dim}: expected (*,{emb_dim}), got {X.shape}")
            continue

        # standardize before manifold learning
        X_std = standardize_columns(X)

        # optional compute-time subsample
        X_compute = subsample_rows(X_std, max_points=compute_max_points, rng=rng)

        print(f"Processing {channel_name} | m={emb_dim} | full={X.shape} | compute={X_compute.shape}")

        # -------------------------
        # UMAP
        # -------------------------
        t0 = time.perf_counter()
        umap_embedding = apply_umap(
            X_compute,
            n_components=n_components_umap,
            n_neighbors=n_neighbors_umap,
            min_dist=min_dist_umap,
            random_state=random_state_umap
        )
        umap_time = time.perf_counter() - t0

        # -------------------------
        # t-SNE
        # -------------------------
        t0 = time.perf_counter()
        tsne_embedding = apply_tsne(
            X_compute,
            n_components=n_components_tsne,
            perplexity=perplexity_tsne,
            random_state=random_state_tsne
        )
        tsne_time = time.perf_counter() - t0

        # -------------------------
        # SAVE EMBEDDINGS
        # -------------------------
        umap_file = f"umap_m{emb_dim}_{channel_name}.npy"
        tsne_file = f"tsne_m{emb_dim}_{channel_name}.npy"

        umap_path = os.path.join(embeddings_out_dir, umap_file)
        tsne_path = os.path.join(embeddings_out_dir, tsne_file)

        np.save(umap_path, umap_embedding)
        np.save(tsne_path, tsne_embedding)

        # -------------------------
        # SUBSAMPLE FOR PLOTTING ONLY
        # -------------------------
        umap_plot = subsample_rows(umap_embedding, max_points=plot_max_points, rng=rng)
        tsne_plot = subsample_rows(tsne_embedding, max_points=plot_max_points, rng=rng)

        # -------------------------
        # UMAP PLOT
        # -------------------------
        fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
        style_ax(ax)
        ax.scatter(
            umap_plot[:, 0], umap_plot[:, 1],
            s=5, color=ACCENT
        )
        ax.set_title(f"UMAP | {channel_name} | m={emb_dim}")
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")
        plt.tight_layout()
        plt.savefig(
            os.path.join(plots_out_dir, f"UMAP_m{emb_dim}_{channel_name}.png"),
            dpi=150,
            bbox_inches="tight"
        )
        plt.close()

        # -------------------------
        # t-SNE PLOT
        # -------------------------
        fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
        style_ax(ax)
        ax.scatter(
            tsne_plot[:, 0], tsne_plot[:, 1],
            s=5, color=ACCENT
        )
        ax.set_title(f"t-SNE | {channel_name} | m={emb_dim}")
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
        plt.tight_layout()
        plt.savefig(
            os.path.join(plots_out_dir, f"tSNE_m{emb_dim}_{channel_name}.png"),
            dpi=150,
            bbox_inches="tight"
        )
        plt.close()

        summary_rows.append({
            "channel": channel_name,
            "emb_dim": emb_dim,
            "n_points_full": X.shape[0],
            "n_points_used_for_compute": X_compute.shape[0],
            "ambient_dim": X.shape[1],
            "umap_time_sec": umap_time,
            "tsne_time_sec": tsne_time,
            "umap_file": umap_file,
            "tsne_file": tsne_file,
        })

        print(
            f"Saved {channel_name} | m={emb_dim} | "
            f"UMAP {umap_embedding.shape} in {umap_time:.2f}s | "
            f"t-SNE {tsne_embedding.shape} in {tsne_time:.2f}s"
        )

# -------------------------------------------------------
# SAVE SUMMARY
# -------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(embeddings_out_dir, "tsne_umap_4to10_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print(f"\nSaved summary CSV: {summary_csv}")
print("Done.")

KeyboardInterrupt: 